In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from collections import Counter, defaultdict
from tqdm.notebook import tqdm
import json
import torch

# Style global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print(f'PyTorch: {torch.__version__} | Device: {"CUDA" if torch.cuda.is_available() else "CPU"}')

PyTorch: 2.10.0+cu130 | Device: CUDA


In [2]:
from datasets import load_dataset

print('Chargement du corpus...')
corpus_ds   = load_dataset('vidore/vidore_v3_pharmaceuticals', 'corpus',   split='test')
queries_ds  = load_dataset('vidore/vidore_v3_pharmaceuticals', 'queries',  split='test')
qrels_ds    = load_dataset('vidore/vidore_v3_pharmaceuticals', 'qrels',    split='test')

# Conversion en DataFrames
df_corpus  = corpus_ds.to_pandas()
df_queries = queries_ds.to_pandas()
df_qrels   = qrels_ds.to_pandas()

Chargement du corpus...


In [3]:
from colpali_engine.models import ColIdefics3, ColIdefics3Processor

# ColSmol-256M — tourne confortablement sur 8GB
MODEL_NAME_B = 'vidore/colsmol-256m'
print(f'Chargement {MODEL_NAME_B}...')

device = "cuda" if torch.cuda.is_available() else "cpu"
model_b = ColIdefics3.from_pretrained(
    MODEL_NAME_B,
    torch_dtype=torch.bfloat16,
    device_map=device,
)
model_b.eval()

processor_b = ColIdefics3Processor.from_pretrained(MODEL_NAME_B)
print(f'Modèle chargé sur {device}')
print(f'VRAM utilisée : {torch.cuda.memory_allocated()/1e9:.2f} Go')

Chargement vidore/colsmol-256m...


Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/420 [00:00<?, ?it/s]

Modèle chargé sur cuda
VRAM utilisée : 0.48 Go


In [4]:
# ── Fonctions d'encodage ──────────────────────────────────────────────────────

def encode_images_colpali(images, model, processor, batch_size=8):
    all_embeddings = []
    for i in tqdm(range(0, len(images), batch_size), desc='Encoding images'):
        batch_imgs = images[i:i+batch_size]
        with torch.no_grad():
            batch_doc = processor.process_images(batch_imgs).to(device)  # CUDA
            embs = model(**batch_doc)
        all_embeddings.extend(list(embs.to(torch.float32).cpu().unbind(0)))
        del embs, batch_doc
        torch.cuda.empty_cache()
    return all_embeddings

def encode_queries_colpali(queries, model, processor, batch_size=16):
    all_embeddings = []
    for i in tqdm(range(0, len(queries), batch_size), desc='Encoding queries'):
        batch_q = queries[i:i+batch_size]
        with torch.no_grad():
            batch_q_proc = processor.process_queries(batch_q).to(device)  # CUDA
            embs = model(**batch_q_proc)
        all_embeddings.extend(list(embs.to(torch.float32).cpu().unbind(0)))
        del embs, batch_q_proc
        torch.cuda.empty_cache()
    return all_embeddings

def colpali_score(query_emb, doc_emb):
    scores = torch.einsum('qd,pd->qp', query_emb, doc_emb)
    return scores.max(dim=1).values.sum().item()

In [5]:
# ── Corpus complet ────────────────────────────────────────────────────────────

MAX_CORPUS_PAGES = 2310
MAX_QUERIES      = 2180

df_corpus_sample  = df_corpus.copy()
valid_corpus_ids  = set(df_corpus_sample['corpus_id'])
valid_qrels       = df_qrels[df_qrels['corpus_id'].isin(valid_corpus_ids)]
valid_query_ids   = set(valid_qrels['query_id'].unique())
df_queries_sample = df_queries[df_queries['query_id'].isin(valid_query_ids)]

print(f'Corpus  : {len(df_corpus_sample):,} pages')
print(f'Queries : {len(df_queries_sample):,} requêtes')
print(f'Qrels   : {len(valid_qrels):,} paires')
print(f'VRAM libre : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.2f} Go')

Corpus  : 2,313 pages
Queries : 2,184 requêtes
Qrels   : 10,392 paires
VRAM libre : 8.11 Go


In [ ]:
from PIL import Image as PILImage
import io
import numpy as np

MAX_IMG_SIZE = 560  # résolution native des patches ColPali — au-delà = perte de temps pure

def safe_to_pil(img, max_size=MAX_IMG_SIZE):
    """Convertit et redimensionne en PIL RGB — 560px = résolution native ColPali."""
    if isinstance(img, dict):
        pil = PILImage.open(io.BytesIO(img['bytes'])).convert('RGB')
    elif isinstance(img, PILImage.Image):
        pil = img.convert('RGB')
    else:
        pil = PILImage.fromarray(np.array(img)).convert('RGB')

    # Redimensionner si nécessaire en gardant le ratio
    w, h = pil.size
    if max(w, h) > max_size:
        ratio = max_size / max(w, h)
        pil = pil.resize((int(w * ratio), int(h * ratio)), PILImage.LANCZOS)

    return pil

In [7]:
import os, pickle
import numpy as np

CACHE_DIR = './colpali_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# Pré-convertir toutes les images en PIL une seule fois (plus rapide qu'iterrows dans la boucle)
print('Pré-conversion PIL (une seule fois)...')
corpus_ids_b   = df_corpus_sample['corpus_id'].tolist()
corpus_images_all = [safe_to_pil(img) for img in tqdm(df_corpus_sample['image'], desc='PIL')]

Pré-conversion PIL (une seule fois)...


PIL:   0%|          | 0/2313 [00:00<?, ?it/s]

In [ ]:
# ── Encodage images ───────────────────────────────────────────────────────────
BATCH_SIZE_IMG = 32
corpus_embs_b  = []
n_batches      = (len(corpus_ids_b) + BATCH_SIZE_IMG - 1) // BATCH_SIZE_IMG

print(f'Encodage images ({len(corpus_ids_b)} pages, batch={BATCH_SIZE_IMG})...')
print(f'Estimation : ~{n_batches} batches')

for i in tqdm(range(0, len(corpus_ids_b), BATCH_SIZE_IMG), total=n_batches):
    cache_path = os.path.join(CACHE_DIR, f'image_{i}.pkl')  # renommé pour éviter conflits

    if os.path.exists(cache_path):
        with open(cache_path, 'rb') as f:
            batch_embs = pickle.load(f)
    else:
        batch_imgs = corpus_images_all[i:i+BATCH_SIZE_IMG]
        with torch.no_grad():
            batch_doc = processor_b.process_images(batch_imgs).to(device)
            embs      = model_b(**batch_doc)
        batch_embs = list(embs.to(torch.float32).cpu().unbind(0))
        del embs, batch_doc
        torch.cuda.empty_cache()
        with open(cache_path, 'wb') as f:
            pickle.dump(batch_embs, f)

    corpus_embs_b.extend(batch_embs)

    # Log toutes les 10 batches
    if (i // BATCH_SIZE_IMG) % 10 == 0:
        vram = torch.cuda.memory_allocated()/1e9
        print(f'  [{len(corpus_embs_b)}/{len(corpus_ids_b)}] VRAM: {vram:.2f} Go')

print(f'Images encodées : {len(corpus_embs_b)}')
print(f'VRAM finale : {torch.cuda.memory_allocated()/1e9:.2f} Go')

Encodage images (2313 pages, batch=32)...
Estimation : ~73 batches


  0%|          | 0/73 [00:00<?, ?it/s]